In [ ]:
#Librerias
from enum import Enum, unique

#Clases
class Point:
  verticeID = 0
  def __init__(self, x: float, y: float):
    Point.verticeID += 1
    self.idVertice = PointverticeID
    self.color = None
    self.x = float(x)
    self.y = float(y)

  def __lt__(self, P): #P son los objetos tipo Point
    if self.y == P.y:
      return self.x < P.x
    return self.y > P.y

  @staticmethod #Permite ejecutar el método directamente desde la clase sin necesidad de instanciar un objeto
  def crossProduct(P1, P2, P3):
    return (P2.x - P1.x) * (P3.y - P1.y) - (P2.y - P1.y) * (P3.x - P1.x)
    #Det positiva: giro izq
    #Det negativa: giro der
    #Det 0: recta

class TipoVertice(Enum):
  START = 1
  END = 2
  SPLIT = 3
  MERGE = 4
  REGULAR = 5

class NodoVertice:
  __slots__ = 'P', 'prev', 'next'
  def __init__(self, P, prev, next):
    self.P = P #Objeto Point
    self.tipoVertice = None
    self.prev = prev
    self.next = next

class MediaArista:
  __slots__ = 'origenP', 'twin', 'next', 'prev', 'trianguloPertenece'
  def __init__(self, origenP):
    self.origenP = origenP
    self.trianguloPertenece = None
    self.twin = None
    self.next = None
    self.prev = None
    #next -> siguiente media arista
    #prev -> media arista anterior
    
class DCEL:
  def __init__(self, verticesP: list, mediasAristas: list, caras: list): #verticesP es una lista de objetos point
    self.verticesP = verticesP
    self.mediasAristas = mediasAristas
    self.caras = caras
  def masDiagonal(self, P1, P2):
    pass

#Algoritmo principal:
class TrianguladorPoligono:
  def __init__(self, poligono, dcel, diagonales, triangulos): #Todos los parametros son listas
    pass

  def clasificarVertices(self):
    pass

  def hacerMonotono(self):
    pass

  def triangularPoligonoMonotono(self, yMonotono):
    pass

  def construyeGraficaDual(self):
    pass

  def asignaColores(self, arbolDual):
    pass

  def seleccionarUbicacion(self, diccPoint):
    pass
#Excepciones: verificar que el objeto Point no exista ya ()


In [ ]:
# ==============================================================================
# ESTRUCTURA DE DATOS: DCEL (Doubly-Connected Edge List)
# ==============================================================================

class Point:
    """Representa una esquina o vértice (x, y) en el plano."""
    def __init__(self, x: float, y: float, index: int = -1):
        self.x = float(x)
        self.y = float(y)
        self.index = index  # Identificador único (v0, v1, v2...)

    def __repr__(self):
        return f"V{self.index}({self.x}, {self.y})"

    def __eq__(self, other):
        return isinstance(other, Point) and self.x == other.x and self.y == other.y


class Face:
    """Representa un triángulo o región interna del polígono."""
    def __init__(self, face_id: int):
        self.face_id = face_id
        self.half_edge = None  # Una semiarista que bordea esta cara

    def __repr__(self):
        return f"Cara_{self.face_id}"


class MediaArista:
    """Representa una semiarista dirigida (Half-Edge)."""
    def __init__(self, origin: Point):
        self.origin = origin          # Punto/Vértice de donde sale la flecha
        self.twin = None              # Flecha gemela en sentido opuesto
        self.next = None              # Siguiente flecha en sentido antihorario
        self.prev = None              # Flecha anterior
        self.incident_face = None     # Cara ubicada a la izquierda de la flecha

    def __repr__(self):
        dest = self.twin.origin if self.twin else "?"
        return f"HalfEdge({self.origin} -> {dest})"


class DCEL:
    """Administra la malla de vértices, medias aristas y caras."""
    def __init__(self):
        self.vertices = []        # Lista de objetos Point
        self.medias_aristas = []  # Lista de objetos MediaArista
        self.caras = []           # Lista de objetos Face
        self.face_count = 0

    def crear_poligono_base(self, coords):
        """Inicializa el polígono simple conectando las paredes en circuito."""
        n = len(coords)
        self.vertices = [Point(x, y, i) for i, (x, y) in enumerate(coords)]

        # Crear la primera cara interna
        f1 = Face(1)
        self.caras.append(f1)

        # Crear semiaristas internas y externas (gemelas)
        edges_in = [MediaArista(v) for v in self.vertices]
        edges_out = [MediaArista(self.vertices[(i + 1) % n]) for i in range(n)]

        for i in range(n):
            e_in = edges_in[i]
            e_out = edges_out[i]

            # 1. Enlazar gemelas (twin)
            e_in.twin = e_out
            e_out.twin = e_in

            # 2. Conectar circuito interno (sentido antihorario)
            e_in.next = edges_in[(i + 1) % n]
            e_in.prev = edges_in[(i - 1) % n]
            e_in.incident_face = f1

            # 3. Conectar circuito externo
            e_out.next = edges_out[(i - 1) % n]
            e_out.prev = edges_out[(i + 1) % n]

            self.medias_aristas.extend([e_in, e_out])

        f1.half_edge = edges_in
        self.face_count = 1

    def masDiagonal(self, v_orig: Point, v_dest: Point):
        """Agrega una diagonal interna entre dos vértices en tiempo constante O(1)."""
        # 1. Buscar las semiaristas que nacen en v_orig y v_dest dentro del recinto
        e_orig = next(e for e in self.medias_aristas if e.origin == v_orig and e.incident_face is not None)
        e_dest = next(e for e in self.medias_aristas if e.origin == v_dest and e.incident_face is not None)

        # 2. Crear las dos nuevas semiaristas gemelas para la diagonal
        d1 = MediaArista(v_orig)
        d2 = MediaArista(v_dest)
        d1.twin = d2
        d2.twin = d1

        # 3. Guardar las referencias anteriores antes de reconectar
        prev_orig = e_orig.prev
        prev_dest = e_dest.prev

        # 4. Reconectar los punteros next y prev para partir la cara en dos
        d1.next = e_dest
        d1.prev = prev_orig
        prev_orig.next = d1
        e_dest.prev = d1

        d2.next = e_orig
        d2.prev = prev_dest
        prev_dest.next = d2
        e_orig.prev = d2

        # 5. Crear una nueva cara y actualizar las pertenencias
        self.face_count += 1
        nueva_cara = Face(self.face_count)
        self.caras.append(nueva_cara)

        # Asignar la cara original al primer circuito
        curr = d1
        while True:
            curr.incident_face = e_orig.incident_face
            curr = curr.next
            if curr == d1:
                break

        # Asignar la nueva cara al segundo circuito
        curr = d2
        while True:
            curr.incident_face = nueva_cara
            curr = curr.next
            if curr == d2:
                break

        d1.incident_face.half_edge = d1
        nueva_cara.half_edge = d2

        self.medias_aristas.extend([d1, d2])
        print(f"✅ Diagonal agregada entre {v_orig} y {v_dest}. Se creó {nueva_cara}.")


# ==============================================================================
# PRUEBA RÁPIDA DE FUNCIONAMIENTO
# ==============================================================================
if __name__ == "__main__":
    dcel = DCEL()
    # Un cuadrado simple de 4 esquinas: (0,0), (2,0), (2,2), (0,2)
    coordenadas = [(0, 0), (2, 0), (2, 2), (0, 2)]
    dcel.crear_poligono_base(coordenadas)

    print(f"Polígono base creado con {len(dcel.vertices)} vértices y {len(dcel.caras)} cara.")

    # Trazamos una diagonal interna de V0 a V2
    dcel.masDiagonal(dcel.vertices, dcel.vertices[1])

    print(f"Total de caras (triángulos) resultantes: {len(dcel.caras)}")